##### Copyright 2026 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini Agents API: Build managed agents with the Interactions API

<a class="tfo-notebook-buttons" target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/quickstarts/Get_started_managed_agents.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

The [Interactions API](https://ai.google.dev/gemini-api/docs/interactions) provides a unified interface for working with Gemini models and agents. The [Getting Started notebook](./Get_started_interactions_api.ipynb) covers how to use it with standard Gemini **models** for text generation, multi-turn conversations, and tool use.

This notebook focuses on something different: **managed agents** with the `antigravity-preview-05-2026` agent.

### `agent=` vs `model=`

When you call the Interactions API, you choose between two modes:

| Parameter | What runs | Best for |
|-----------|-----------|----------|
| `model="gemini-..."` | A standard Gemini model | Text generation, structured output, function calling |
| `agent="antigravity-preview-05-2026"` | A **managed agent** in a sandboxed Linux environment | Autonomous tasks: code execution, web research, file management |

With `model=`, you get a stateless LLM call (see the [Getting Started notebook](./Get_started_interactions_api.ipynb)). With `agent=`, you spin up an autonomous agent that can **reason, plan, write and execute code, browse the web, and manage files** — all inside a secure sandbox, without you writing any orchestration logic.

This notebook walks you through the agent mode step by step:

1. **Simple questions** — use the agent like an LLM (it works, but it's overkill!)
2. **Multi-turn conversations** — persistent sandbox = built-in memory
3. **Using tools** — code execution, web search, file operations
4. **Loading data into the sandbox** — inject files before the agent starts
5. **Creating reusable custom agents** — bundle instructions, skills, and environment

<a name="setup"></a>
## Setup

### Install SDK

Install the SDK from [PyPI](https://github.com/googleapis/python-genai). It's recommended to always use the latest version.

In [1]:
%pip install -U -q "google-genai>=2.9.0"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 10.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


### Setup your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GEMINI_API_KEY`. If you don't already have an API key or you aren't sure how to create a Colab Secret, see [Authentication ![image](https://storage.googleapis.com/generativeai-downloads/images/colab_icon16.png)](../quickstarts/Authentication.ipynb) for an example.

In [29]:
from google.colab import userdata

GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')

### Initialize SDK client

With the new SDK, now you only need to initialize a client with you API key.

In [4]:
import uuid
from google import genai
from google.genai import types
from IPython.display import Markdown

client = genai.Client(api_key=GEMINI_API_KEY)

# The default managed agent.
AGENT = "antigravity-preview-05-2026"

# Generate a unique suffix for this notebook session to prevent agent ID conflicts
UNIQUE_SUFFIX = uuid.uuid4().hex[:8]

print("Client ready!")

Client ready!


## 1. Simple questions — the agent as an LLM

The simplest way to use a managed agent is to ask it a question, just like you'd call a standard Gemini model. Pass `agent="antigravity-preview-05-2026"` and `environment="remote"` to create a fresh Linux sandbox for the agent.

This works, but it's a bit like driving a Formula 1 car to the grocery store — the agent has code execution, web search, and file management capabilities that are all sitting idle for a simple factual question.

In [5]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What is the capital of France?",
    environment="remote",
)

Markdown(interaction.output_text)

The capital of France is Paris.

In [6]:
# The response also includes metadata about the agent's sandbox.
print(f"Status:         {interaction.status}")
print(f"Interaction ID: {interaction.id}")
print(f"Environment ID: {interaction.environment_id}")

Status:         completed
Interaction ID: v1_ChdCZFNOYXRmY0E0R2RqTWNQcFlhWHNRaxIXQmRTTmF0ZmNBNEdkak1jUHBZYVhzUWs
Environment ID: 142ebd47e606645fa32cae55d2e58eec


Notice the `environment_id` in the response. That's the agent's persistent Linux sandbox. Even for this simple question, a full container was provisioned. Let's make use of that persistence next.

## 2. Multi-turn conversations

Since each agent runs in a persistent sandbox, you can **continue where you left off** by reusing the `environment_id` and linking turns with `previous_interaction_id`.

This is fundamentally different from stateless `model=` calls. The agent has a true *persistent environment* — files it creates stick around, packages it installs remain available, and conversation context is preserved.

In [7]:
# Turn 1: Introduce yourself.
turn1 = client.interactions.create(
    agent=AGENT,
    input="Hi! My name is Alice and I'm a software engineer. Remember that in a knowledge.md doc.",
    environment= "remote",
)

Markdown(f"**Turn 1:** {turn1.output_text}")

**Turn 1:** Nice to meet you, Alice! I have saved your information in `knowledge.md`. Let me know if you need help with anything!

In [9]:
# Turn 2: Ask whether the agent remembers.
# Pass environment_id and previous_interaction_id to continue the conversation.
turn2 = client.interactions.create(
    agent=AGENT,
    input="what's my name and what do I do?",
    environment= turn1.environment_id,
    previous_interaction_id= turn1.id,
)

Markdown(f"**Turn 2:** {turn2.output_text}")

**Turn 2:** Your name is Alice, and you are a software engineer!

The agent remembered across turns because you passed `environment` with the previous environment ID — same sandbox, which means same files.

You could have achieved the same result using `previous_interaction_id` to keep the history of the previous conversation, but that would not have showcased the environement specificities.

This is how you build stateful, multi-turn workflows. See the [Getting Started notebook](./Get_started_interactions_api.ipynb) for `model=`-based multi-turn using `previous_interaction_id` alone (without environments).

## 3. Using tools — where the agent shines

This is where managed agents go beyond a standard chat model. The antigravity-preview-05-2026 agent has **built-in tools** it uses autonomously — you don't declare them, just describe your goal and the agent figures out what to use.

| Tool | Description |
|------|-------------|
| `bash` | Execute shell commands in the sandbox |
| `google_search` | Search the web for current information |
| `url_context` | Fetch and extract text from URLs |
| `write_file` | Create or overwrite files in the sandbox |
| `read_file` | Read file contents from the sandbox |
| `list_files` | List directory contents |
| `delete_file` | Remove files from the sandbox |

For the standard `model=`-based tools (Google Search grounding, code execution, function calling), see the [Getting Started notebook](./Get_started_interactions_api.ipynb) and the dedicated tool notebooks:
- [Code Execution](./Code_Execution.ipynb)
- [Search Grounding](./Search_Grounding.ipynb)
- [Function Calling](./Function_calling.ipynb)

### Code execution

Ask a computational question and the agent will write code, run it in its sandbox, and return the verified result.

In [10]:
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Write a Python script that computes the first 20 Fibonacci numbers. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

Here is the Python script to compute the first 20 Fibonacci numbers:

### `fibonacci.py`
```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    numbers = fibonacci(n)
    print(f"First {n} Fibonacci numbers:")
    print(numbers)
```

### Execution Output
```text
First 20 Fibonacci numbers:
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]
```

### Inspecting steps — what the agent actually did

The `steps` field in the response shows the agent's reasoning chain: its thoughts, tool calls, tool results, and final output. This is useful for debugging and understanding the agent's behavior.

In [11]:
# Inspect the steps from the Fibonacci interaction above.
for i, step in enumerate(interaction.steps):
    step_type = step.type
    print(f"--- Step {i} [{step_type}] ---")

    # Tool call steps show which tool was invoked and with what arguments.
    if hasattr(step, "name") and step.name:
        print(f"  Tool: {step.name}")
        if hasattr(step, "arguments"):
            args_str = str(step.arguments)[:300]
            print(f"  Args: {args_str}")

    # Content steps contain the agent's text output.
    if hasattr(step, "content") and step.content:
        for c in step.content:
            if hasattr(c, "text"):
                print(f"  Text: {c.text[:300]}")
    print()

--- Step 0 [code_execution_call] ---

--- Step 1 [code_execution_result] ---

--- Step 2 [function_call] ---
  Tool: write_file
  Args: {'toolAction': 'Writing Fibonacci script to file', 'content': 'def fibonacci(n):\n    fib_series = []\n    a, b = 0, 1\n    for _ in range(n):\n        fib_series.append(a)\n        a, b = b, a + b\n    return fib_series\n\nif __name__ == "__main__":\n    n = 20\n    numbers = fibonacci(n)\n    prin

--- Step 3 [function_result] ---
  Tool: write_file

--- Step 4 [code_execution_call] ---

--- Step 5 [code_execution_result] ---

--- Step 6 [model_output] ---
  Text: Here is the Python script to compute the first 20 Fibonacci numbers:

### `fibonacci.py`
```python
def fibonacci(n):
    fib_series = []
    a, b = 0, 1
    for _ in range(n):
        fib_series.append(a)
        a, b = b, a + b
    return fib_series

if __name__ == "__main__":
    n = 20
    number



### Web search

The agent can search the web autonomously when it needs up-to-date information.

In [12]:
interaction = client.interactions.create(
    agent=AGENT,
    input="What were the top 3 news stories about Google this week? Summarize them briefly.",
    environment="remote",
)

Markdown(interaction.output_text)

Here are the top three news stories regarding Google this week:

---

### 1. Federal Court Approves $700 Million Google Play Antitrust Settlement
* **What Happened:** A federal judge granted final approval to a **$700 million settlement** resolving a major multi-state antitrust lawsuit led by attorneys general across all 50 U.S. states and territories [1.1, 1.2].
* **Key Details:** The lawsuit alleged anticompetitive conduct and monopoly power regarding Google Play Store app distribution and in-app billing practices. The settlement directs the vast majority of funds as direct refunds to consumers who made Play Store purchases between August 2016 and September 2023. Additionally, Google agreed to business practice reforms for the next 5 to 7 years, including enabling alternative in-app billing systems, allowing developers to communicate outside payment options, and supporting third-party app stores on Android [1.1, 1.2].

---

### 2. Google Completes the August 2026 Search Spam Algorithm Update
* **What Happened:** Google launched and finalized its **August 2026 Spam Update**, completing the global rollout in just 2 days and 16 hours [1.3, 1.4].
* **Key Details:** Marking the third spam update of 2026 (following updates in March and June), this release applied across all languages and regions. Powered by enhancements to Google's *SpamBrain* automated detection systems, the update targeted policy-violating search manipulation tactics, low-quality scaled automated content, and spammy SEO techniques to improve organic Search quality [1.3, 1.5].

---

### 3. Google Expands "Preferred Sources" & AI Personalization Features
* **What Happened:** Google rolled out enhanced personalization features across Search, Discover, and Google News, centered on an upgraded **"Preferred Source"** system [1.6, 1.7].
* **Key Details:** Web publishers can now integrate an interactive, embeddable one-click button directly on their websites that allows readers to mark them as a preferred source without leaving the page [1.7, 1.8]. Once favorited, stories from those publications are prioritized and badged in Google's *Top Stories* module as well as integrated into Gemini-generated *AI Overviews* and *AI Mode* search summaries [1.6, 1.8].

---

### Sources
* [1] [Washington State AGO – $700 Million Google Play Settlement](https://www.atg.wa.gov/news/news-releases/wa-bipartisan-coalition-secure-700-million-google-settlement-over-app-store)
* [2] [First Coast News – Judge Approves Settlement in Google Play Store Monopoly Lawsuit](https://www.firstcoastnews.com/article/news/nation-world/judge-approves-settlement-google-play-store-monopoly-lawsuit/507-bd16aaac-b46e-4b73-958f-fd4431619747)
* [3] [Search Engine Journal – Google Finishes Rolling Out the August 2026 Spam Update](https://www.searchenginejournal.com/google-begins-rolling-out-the-august-2026-spam-update/586301/)
* [4] [Search Engine Land – Google August 2026 Spam Update Done Rolling Out](https://searchengineland.com/google-august-2026-spam-update-done-rolling-out-485471)
* [5] [Search Engine Roundtable – Google August 2026 Spam Update Is Rolling Out](https://www.seroundtable.com/google-august-2026-spam-update-41895.html)
* [6] [Google Blog – Personalize the Content You See on Search, Discover, and News](https://blog.google/products-and-platforms/products/search/personalize-search-discover-news/)
* [7] [Search Engine Land – Google Makes the Preferred Source Button More Seamless](https://searchengineland.com/google-makes-the-preferred-source-button-more-seamless-485548)
* [8] [Press Gazette – One-Click Link in Articles Adds Publishers to Google 'Preferred Source'](https://pressgazette.co.uk/platforms/google-preferred-source-article-users-curate-sources-top-stories-ai-overview-search/)

### File operations

The agent can create, read, and manage files in its sandbox. Files persist within the environment across turns.

In [13]:
# Ask the agent to create a file, run it, and show results.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a Python file called 'analysis.py' that generates 50 random numbers, "
        "computes mean, median, and standard deviation, then prints the results. "
        "Run it and show the output."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

I have created `analysis.py` and executed it.

### Script (`analysis.py`)

```python
import random
import statistics

# Generate 50 random numbers
random.seed(42)  # Seed set for reproducible results
data = [random.uniform(1, 100) for _ in range(50)]

mean_val = statistics.mean(data)
median_val = statistics.median(data)
stdev_val = statistics.stdev(data)

print("Generated 50 random numbers:")
print([round(x, 2) for x in data])
print("\n--- Summary Statistics ---")
print(f"Mean:               {mean_val:.4f}")
print(f"Median:             {median_val:.4f}")
print(f"Standard Deviation: {stdev_val:.4f}")
```

---

### Output

```
Generated 50 random numbers:
[64.3, 3.48, 28.23, 23.1, 73.91, 67.99, 89.33, 9.61, 42.77, 3.95, 22.65, 51.03, 3.63, 20.68, 65.34, 54.95, 22.82, 59.34, 81.13, 1.64, 80.78, 70.12, 34.68, 16.39, 95.76, 34.32, 10.18, 10.57, 84.9, 60.77, 80.91, 73.24, 54.09, 97.34, 38.47, 55.65, 83.11, 62.23, 86.31, 58.16, 70.75, 5.54, 23.56, 29.65, 8.9, 24.05, 11.0, 28.52, 63.93, 37.12]

--- Summary Statistics ---
Mean:               45.6177
Median:             46.9002
Standard Deviation: 29.0356
```

## 4. Loading data into the agent's sandbox

You can inject files into the agent's environment **before it starts** using `sources`. This is how you provide data, configuration, or code for the agent to work with.

| Source type | Description | Best for |
|------------|-------------|----------|
| `inline` | Embed content directly (max 75 KB) | Config files, small scripts |
| `gcs` | Load from Google Cloud Storage | Large datasets |
| `repository` | Load from GitHub | Code repositories |

In [14]:
# Inject a CSV file inline and ask the agent to analyze it.
csv_data = """name,age,city,score
              Alice,28,Paris,92
              Bob,35,London,87
              Charlie,42,Berlin,95
              Diana,31,Tokyo,88
              Eve,26,Sydney,91"""

interaction = client.interactions.create(
    agent=AGENT,
    input="Read the file data.csv, analyze it, and tell me who scored the highest.",
    environment={
        "type": "remote",
        "sources": [
            {
                "type": "inline",
                "content": csv_data,
                "target": "/workspace/data.csv",
            }
        ],
    },
)

Markdown(interaction.output_text)

Based on `data.csv`, here is the breakdown of the scores:

| Name | Age | City | Score |
| :--- | :--- | :--- | :--- |
| Alice | 28 | Paris | 92 |
| Bob | 35 | London | 87 |
| **Charlie** | **42** | **Berlin** | **95** |
| Diana | 31 | Tokyo | 88 |
| Eve | 26 | Sydney | 91 |

**Charlie** scored the highest with a score of **95**.

You can also load from other sources:

```python
# From Google Cloud Storage
{"type": "gcs", "source": "gs://my-bucket/data/", "target": "/workspace/data/"}

# From a GitHub repository
{"type": "repository", "source": "https://github.com/user/repo", "target": "/workspace/repo/"}
```

You can combine multiple sources in a single request — the agent will have access to all of them at startup.

**Pro tip:** You can use that to add skills to you agent, as you'll see next.

## 5. Creating reusable custom agents

So far, every interaction has used the base `antigravity-preview-05-2026` agent with inline instructions. Once you've found a setup that works well, you can **persist it into a named custom agent** that bundles:

- **Instructions** — system prompt that defines the agent's behavior
- **Environment** — pre-configured sandbox with files and sources
- **Skills** — `SKILL.md` files that teach the agent specialized capabilities

This is the recommended workflow:
1. **Prototype** with `agent="antigravity-preview-05-2026"` — iterate on instructions, sources, and prompts
2. **Create** a named agent via the `/agents` endpoint
3. **Invoke** your agent by name from any client

### Creating a custom agent

In [15]:
# Create a custom data analysis agent using the SDK.
my_agent = client.agents.create(
    id=f"my-data-analyst-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a data analysis assistant. "
        "Always write Python code using pandas to answer questions. "
        "Show your code and output clearly. "
        "When creating visualizations, save them as PNG files."
    ),
    base_environment={
        "type": "remote",
    },
)

print(f"✓ Agent created: {my_agent.id}")

✓ Agent created: my-data-analyst-72c4a5dd


/tmp/ipykernel_2217/1261841246.py:2: UserWarning: Agents usage is experimental and may change in future versions.
  my_agent = client.agents.create(


### Using a custom agent

Once created, invoke your agent by name. It will follow its instructions automatically.

In [16]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-data-analyst-{UNIQUE_SUFFIX}",
    input=(
        "Generate a sample dataset of 100 sales records with columns: "
        "product, region, revenue, quantity. "
        "Find the top 5 products by total revenue and show the analysis."
    ),
    environment="remote",
)

Markdown(interaction.output_text)

A 100-record sales dataset was generated with realistic pricing and regional distribution. The analysis was conducted using Python and `pandas`, and a visualization was saved as `top_5_products_revenue.png`.

---

### Python Code

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set random seed for reproducibility
np.random.seed(42)

# Define categories
products = [
    'Laptop', 'Smartphone', 'Wireless Headphones', 'Smartwatch', 
    'Tablet', 'Monitor', 'Mechanical Keyboard', 'Gaming Mouse', 
    'USB-C Dock', 'External SSD'
]
regions = ['North America', 'Europe', 'Asia-Pacific', 'Latin America']

# Generate 100 random sales records
num_records = 100
sample_products = np.random.choice(products, size=num_records)
sample_regions = np.random.choice(regions, size=num_records)
sample_quantities = np.random.randint(1, 25, size=num_records)

# Base unit prices for realistic revenues
unit_prices = {
    'Laptop': 950,
    'Smartphone': 700,
    'Tablet': 450,
    'Monitor': 300,
    'Smartwatch': 220,
    'Wireless Headphones': 150,
    'External SSD': 120,
    'USB-C Dock': 90,
    'Mechanical Keyboard': 80,
    'Gaming Mouse': 50
}

# Calculate revenue with random price variation (+/- 5%)
sample_revenues = [
    round(sample_quantities[i] * unit_prices[sample_products[i]] * np.random.uniform(0.95, 1.05), 2)
    for i in range(num_records)
]

# Create DataFrame
df = pd.DataFrame({
    'product': sample_products,
    'region': sample_regions,
    'revenue': sample_revenues,
    'quantity': sample_quantities
})

# Save dataset
df.to_csv('sales_data.csv', index=False)

# Analysis: Top 5 products by total revenue
product_summary = df.groupby('product').agg(
    total_revenue=('revenue', 'sum'),
    total_quantity=('quantity', 'sum'),
    order_count=('revenue', 'count'),
    avg_order_value=('revenue', 'mean')
).sort_values(by='total_revenue', ascending=False)

top_5_products = product_summary.head(5).reset_index()

# Regional breakdown for top 5 products
top_5_names = top_5_products['product'].tolist()
top_5_regional = (
    df[df['product'].isin(top_5_names)]
    .groupby(['product', 'region'])['revenue']
    .sum()
    .unstack()
    .fillna(0)
    .loc[top_5_names]
)

# Visualization
plt.figure(figsize=(10, 6))
sns.set_theme(style="whitegrid")

barplot = sns.barplot(
    data=top_5_products,
    x='product',
    y='total_revenue',
    hue='product',
    palette='Blues_r',
    legend=False
)

plt.title('Top 5 Products by Total Revenue', fontsize=15, fontweight='bold', pad=15)
plt.xlabel('Product', fontsize=12, labelpad=10)
plt.ylabel('Total Revenue ($)', fontsize=12, labelpad=10)
plt.xticks(rotation=15)

for p in barplot.patches:
    height = p.get_height()
    barplot.annotate(
        f"${height:,.2f}",
        (p.get_x() + p.get_width() / 2., height),
        ha='center', va='bottom',
        xytext=(0, 4), textcoords='offset points',
        fontsize=10, fontweight='bold'
    )

plt.tight_layout()
plt.savefig('top_5_products_revenue.png', dpi=300)
plt.close()
```

---

### Output & Analysis

#### 1. Sample Records (First 10 of 100)
| Product | Region | Revenue ($) | Quantity |
| :--- | :--- | :--- | :--- |
| Mechanical Keyboard | Latin America | 81.14 | 1 |
| Smartwatch | Europe | 4,006.17 | 19 |
| Gaming Mouse | North America | 483.08 | 10 |
| Tablet | Latin America | 5,615.22 | 12 |
| Mechanical Keyboard | Asia-Pacific | 1,940.43 | 24 |
| External SSD | Asia-Pacific | 1,711.66 | 15 |
| Wireless Headphones | Europe | 3,168.49 | 22 |
| Mechanical Keyboard | Latin America | 1,951.39 | 24 |
| Gaming Mouse | North America | 427.73 | 9 |
| Tablet | Asia-Pacific | 8,694.73 | 20 |

---

#### 2. Top 5 Products by Total Revenue

| Rank | Product | Total Revenue ($) | Total Quantity | Order Count | Avg Order Value ($) |
| :---: | :--- | :---: | :---: | :---: | :---: |
| **1** | **Laptop** | **$106,812.27** | 114 | 7 | $15,258.90 |
| **2** | **Smartphone** | **$73,809.04** | 103 | 10 | $7,380.90 |
| **3** | **Tablet** | **$70,511.46** | 154 | 10 | $7,051.15 |
| **4** | **Smartwatch** | **$26,898.68** | 122 | 9 | $2,988.74 |
| **5** | **External SSD** | **$19,373.85** | 162 | 11 | $1,761.26 |

---

#### 3. Regional Revenue Breakdown for Top 5 Products

| Product | Asia-Pacific ($) | Europe ($) | Latin America ($) | North America ($) |
| :--- | :---: | :---: | :---: | :---: |
| **Laptop** | $36,811.01 | $21,943.18 | $0.00 | $48,058.08 |
| **Smartphone** | $16,673.95 | $23,576.97 | $29,388.28 | $4,169.84 |
| **Tablet** | $27,097.22 | $21,238.92 | $10,718.73 | $11,456.59 |
| **Smartwatch** | $10,175.37 | $4,006.17 | $8,650.05 | $4,067.09 |
| **External SSD** | $3,964.87 | $3,236.05 | $4,689.05 | $7,483.88 |

---

### Key Takeaways
- **Top Performer:** **Laptop** generated the highest total revenue ($106,812.27 across 7 orders) due to its high unit value.
- **Volume vs. Revenue:** **External SSD** sold the highest quantity among the top five (162 units), but ranks 5th in revenue due to lower unit pricing.
- **Top Region for Laptops:** **North America** led laptop revenue ($48,058.08), followed by **Asia-Pacific** ($36,811.01).
- The generated chart has been saved as `top_5_products_revenue.png`.

### Creating an agent with pre-loaded data

You can define the agent's environment with sources, so data is ready before the agent starts:

In [17]:
# Create an agent with GCS sources pre-loaded using the SDK.
my_slides_agent = client.agents.create(
    id=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    base_agent=AGENT,
    system_instruction=(
        "You are a software engineer speciliazed in the Gemini API. "
        "Use the skills available in /.agents/skills/ to create amazing apps."
    ),
    base_environment={
        "type": "remote",
        "sources": [
            {
                "type": "repository",
                "source": "https://github.com/google-gemini/gemini-skills",
                "target": "/.agents/skills",
            }
        ],
    },
)

print(f"✓ Agent created: {my_slides_agent.id}")

✓ Agent created: my-gemini-api-agent-72c4a5dd


In [18]:
# Invoke the custom agent.
interaction = client.interactions.create(
    agent=f"my-gemini-api-agent-{UNIQUE_SUFFIX}",
    input="Tell me what you can do with your skills?",
    environment="remote",
)

Markdown(interaction.output_text)

With the specialized Gemini API skills available, I can help you design, build, and deploy a wide range of AI-powered applications across Python, JavaScript/TypeScript, Go, and Java. 

Here is an overview of what I can do:

---

### 1. **Core Gemini API & Multimodal Development** (`gemini-api-dev`)
* **Multimodal Processing:** Ingest and analyze combinations of text, images, audio, and video files.
* **Structured Outputs & Schema Enforcement:** Generate structured data (such as JSON or typed models) adhering to strict schemas (e.g., Pydantic / Zod).
* **Tool & Function Calling:** Connect Gemini models to external APIs, databases, and custom functions with automatic tool calling or manual orchestration.
* **Model Selection & Configuration:** Choose and optimize settings (temperature, top_p, system instructions, safety settings) across Gemini and Gemma model families.

---

### 2. **Interactions API & Agent Workflows** (`gemini-interactions-api`)
* **Conversational & Multi-Turn Chat:** Build stateful chat applications, agentic workflows, and background research tasks.
* **Streaming & Real-Time Responses:** Implement low-latency streaming outputs for text and code generation.
* **Generation Workflows:** Coordinate complex multi-step reasoning, image generation, and structured synthesis.
* **API Migration:** Assist in upgrading and modernizing legacy `generateContent` implementations to current SDK standards.

---

### 3. **Real-Time Bidirectional Streaming** (`gemini-live-api-dev`)
* **WebSocket-Based Live Streaming:** Build low-latency, real-time voice and video conversational agents.
* **Voice Activity Detection (VAD) & Native Audio:** Configure dynamic audio input/output, interruption handling, and voice selection.
* **Ephemeral Authentication:** Set up secure, temporary client tokens for direct browser or mobile app connections without exposing API keys.
* **Real-Time Multimodal Features:** Enable live video feed analysis, camera streaming, and real-time translation.

---

### 4. **Generative Video & Media Processing** (`gemini-omni-flash-api`)
* **Text-to-Video & Image-to-Video:** Generate new video clips from text prompts or reference/starting images.
* **Generative Video Editing & Transitions:** Create turn-by-turn video edits, seamless transitions from first/last frames, and video variations.
* **Media Pre-Processing:** Optimize high-resolution video and audio using `ffmpeg` (e.g., resizing, segmenting, audio stripping/remixing) before API submission.

---

### How We Can Get Started
Let me know what you want to build—whether it's a full-stack Next.js/Python web app, a real-time voice agent, a multimodal analysis pipeline, or a generative media tool—and I can generate the code, set up the project structure, and help you deploy it.

### Forking from an existing environment

If you've already set up a sandbox you like (installed packages, created files, etc.), you can fork it into a new agent using the `environment_id` from a previous interaction:

```python
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="Your custom instructions here.",
    base_environment={"env_id": "YOUR_ENVIRONMENT_ID"},
)
```

This captures the exact state of that sandbox — all installed packages, files, and configuration.

In [19]:
my_forked_agent = client.agents.create(
    id="my-forked-agent",
    base_agent=AGENT,
    system_instruction="I want all your apps to use the Live API",
    base_environment={"env_id": interaction.environment_id},
)
print(f"✓ Agent forked: {my_forked_agent.id}")

✓ Agent forked: my-forked-agent


### Managing agents (CRUD)

The `/agents` endpoint supports full lifecycle management:

In [21]:
# List all your agents.
print("Your agents:")
for agent in client.agents.list().agents:
    print(f"- {agent.id}")

# Get a specific agent's details.
agent = client.agents.get(id=f"my-data-analyst-{UNIQUE_SUFFIX}")
print(f"\nAgent details for {agent.id}:")
print(f"Base agent: {agent.base_agent}")
print(f"System instruction: {agent.system_instruction}")

Your agents:
- my-data-analyst-72c4a5dd
- my-forked-agent
- my-gemini-api-agent-72c4a5dd

Agent details for my-data-analyst-72c4a5dd:
Base agent: antigravity-preview-05-2026
System instruction: You are a data analysis assistant. Always write Python code using pandas to answer questions. Show your code and output clearly. When creating visualizations, save them as PNG files.


In [22]:
# Clean up: delete the agents you created.
for agent_name in [f"my-data-analyst-{UNIQUE_SUFFIX}", f"my-forked-agent-{UNIQUE_SUFFIX}", f"my-gemini-api-agent-{UNIQUE_SUFFIX}"]:
    try:
        client.agents.delete(id=agent_name)
        print(f"✓ Deleted {agent_name}")
    except Exception as e:
        print(f"  Failed to delete {agent_name}: {e}")

✓ Deleted my-data-analyst-72c4a5dd
✓ Deleted my-forked-agent-72c4a5dd
✓ Deleted my-gemini-api-agent-72c4a5dd


### Agent directory structure

Behind the API, an agent is defined by a simple set of files. This is what gets deployed when you create one:

```
my-agent/
├── agent.yaml       # Configuration: base agent, tools, environment
├── AGENTS.md        # System instructions (loaded automatically)
├── skills/          # Custom SKILL.md files that extend capabilities
└── workspace/       # Files seeded into the remote sandbox at startup
```

- **`agent.yaml`** maps directly to the `/agents` API resource
- **`AGENTS.md`** provides system instructions — automatically loaded by the harness
- **`skills/`** contains specialized `SKILL.md` files the agent discovers and uses
- **`workspace/`** files are injected into the sandbox at startup

This file-based structure makes agents easy to version-control, share, and iterate on. Check the [documentation](https://ai.google.dev/gemini-api/docs/custom-agents#file-based_customization) for more details.

## 6. Streaming

For longer tasks, enable streaming with `stream=True` to get real-time updates as the agent works. Instead of waiting for the complete response, you receive a stream of **Server-Sent Events (SSE)** that let you show progress to the user.

### Event types

The stream delivers events that tell you what the agent is doing:

| Event type | Meaning | What to do |
|------------|---------|------------|
| `interaction.created` | The interaction was created | Store the `id` for later reference |
| `interaction.status_update` | Status changed (e.g., `in_progress`) | Update UI status indicator |
| `step.start` | A new step began (thinking, tool call, output) | Show a loading indicator |
| `step.delta` | Incremental content — a chunk of text, thought, or tool output | **Append to display** — this is the main content |
| `step.stop` | A step completed | Hide loading indicator |
| `interaction.completed` | The agent finished all work | Finalize the UI |

The `step.delta` events are where the content lives. Each delta has a `type` (e.g., `text`, `thought`, `function_call`, `function_result`) and content you can render incrementally.

In [23]:
# Stream a response and collect the text as it arrives.
stream = client.interactions.create(
    agent=AGENT,
    input="Write a short poem about the ocean.",
    stream=True,
    environment="remote",
)

collected_text = []

for event in stream:
    # Show the event type so you can see the lifecycle.
    if event.event_type in ("interaction.created", "step.start", "step.stop", "interaction.completed"):
        print(f"[{event.event_type}]")

    # step.delta events carry the actual content.
    elif event.event_type == "step.delta":
        delta = event.delta
        if hasattr(delta, "text") and delta.text:
            print(delta.text, end="", flush=True)
            collected_text.append(delta.text)

print(f"\n\n--- Collected {len(collected_text)} text chunks ---")

[interaction.created]
[step.start]
Boundless expanse of rolling blue,  
Where sunbeams dance and tides renew.  
Beneath the foam and gentle breeze,  
Lie silent depths and ancient seas.  
A rhythmic breath upon the shore,  
In motion now and evermore.[step.stop]
[interaction.completed]


--- Collected 2 text chunks ---


## 7. Advanced features

### Network configuration

By default, the agent's sandbox has unrestricted outbound network access. You can control this with the `network` field:
- **Allowlist specific domains** — only requests to listed domains are permitted
- **Inject credentials** — automatically add headers (API keys, tokens) to outbound requests
- **Disable network** — set `network: "disabled"` to block all outbound traffic

In [31]:
# Allow the agent to call only the Gemini API, with an auto-injected API key.
interaction = client.interactions.create(
    agent=AGENT,
    input="Use curl to call the Gemini API and list available models. Show the first 3.",
    environment={
        "type": "remote",
        "network": {
            "allowlist": [
                {
                    "domain": "generativelanguage.googleapis.com",
                    "transform": {"x-goog-api-key": GEMINI_API_KEY},
                },
            ]
        },
    },
)

Markdown(interaction.output_text)

Here is the `curl` command used to query the Gemini API:

```bash
curl -s https://generativelanguage.googleapis.com/v1beta/models | jq '.models[:3]'
```

### Result (First 3 Models)

```json
[
  {
    "name": "models/gemini-2.5-flash",
    "version": "001",
    "displayName": "Gemini 2.5 Flash",
    "description": "Stable version of Gemini 2.5 Flash, our mid-size multimodal model that supports up to 1 million tokens, released in June of 2025.",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-pro",
    "version": "2.5",
    "displayName": "Gemini 2.5 Pro",
    "description": "Stable release (June 17th, 2025) of Gemini 2.5 Pro",
    "inputTokenLimit": 1048576,
    "outputTokenLimit": 65536,
    "supportedGenerationMethods": [
      "generateContent",
      "countTokens",
      "createCachedContent",
      "batchGenerateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2,
    "thinking": true
  },
  {
    "name": "models/gemini-2.5-flash-preview-tts",
    "version": "gemini-2.5-flash-exp-tts-2025-05-19",
    "displayName": "Gemini 2.5 Flash Preview TTS",
    "description": "Gemini 2.5 Flash Preview TTS",
    "inputTokenLimit": 8192,
    "outputTokenLimit": 16384,
    "supportedGenerationMethods": [
      "countTokens",
      "generateContent"
    ],
    "temperature": 1,
    "topP": 0.95,
    "topK": 64,
    "maxTemperature": 2
  }
]
```

### Download environment snapshots

You can download all the files the agent created or modified as a tar archive. This lets you retrieve the agent's work products — code, data, reports — from the sandbox.

In [32]:
import subprocess
import tarfile
import os

# Create an interaction where the agent produces files.
interaction = client.interactions.create(
    agent=AGENT,
    input=(
        "Create a directory called 'project' with a README.md and a hello.py script. "
        "List the files you created."
    ),
    environment="remote",
)

env_id = interaction.environment_id
print(f"Environment ID: {env_id}")

Markdown(interaction.output_text)

Environment ID: 469523211d38a5e3fc2e655f4f619eea


The directory `project` has been created with the requested files.

### Created Files in `project`:
- `README.md`
- `hello.py`

In [33]:
# Download the environment snapshot.
download_url = (
    f"https://generativelanguage.googleapis.com/v1beta/"
    f"files/environment-{env_id}:download?alt=media"
)

result = subprocess.run(
    ["curl", "-L", "-s", "-o", "snapshot.tar",
     "-H", f"x-goog-api-key: {GEMINI_API_KEY}",
     download_url],
    capture_output=True, text=True,
)

if os.path.exists("snapshot.tar") and os.path.getsize("snapshot.tar") > 0:
    with tarfile.open("snapshot.tar") as tar:
        print("Files in snapshot:")
        for member in tar.getmembers():
            print(f"  {member.name} ({member.size} bytes)")
else:
    print("Snapshot not available (environment may have expired).")

Files in snapshot:
  . (0 bytes)
  ./project (0 bytes)
  ./project/README.md (47 bytes)
  ./project/hello.py (78 bytes)


## Next steps

You've walked through the core capabilities of managed agents:

1. ✅ **Simple Q&A** — the agent can answer questions like an LLM
2. ✅ **Multi-turn** — persistent sandbox enables stateful conversations
3. ✅ **Built-in tools** — code execution, web search, file management
4. ✅ **Data loading** — inject files via inline, GCS, or GitHub sources
5. ✅ **Custom agents** — reusable configurations with instructions, skills, and environment
6. ✅ **Streaming** — real-time updates as the agent works
7. ✅ **Advanced** — network control and environment snapshots

### Learn more

- **[Getting Started notebook](./Get_started_interactions_api.ipynb)** — the `model=`-based Interactions API for standard generation, multi-turn, and tools
- **[Managed Agents documentation](https://ai.google.dev/gemini-api/docs/eap/gemini-agents/gemini-agents)** — full reference for the agent API
- **[Code Execution](./Code_Execution.ipynb)** — model-based code execution
- **[Search Grounding](./Search_Grounding.ipynb)** — model-based web search
- **[Function Calling](./Function_calling.ipynb)** — custom function declarations